## FCLGA GraphTransformer: Linear Demo

This notebook demonstrated the complete FCLGA workflow for CFRP Strain Field Prediction on a Linear Elastic Laminate.

1. **Environment Setup** Verify the conda environment is activated and ready.
2. **Preprocessing pipeline:** 6-step pipeline generating 500 parametric samples.
3. **Training:** Hyperparameter optimized training.  
4. **Testing:** Evaluation on test set. 
5. **Results:** Display of strain field predictions.


## Step 1: Environment Setup

In [1]:
# Verify environment
import sys
print(f"Python: {sys.version}")
print(f"Conda environment: {sys.prefix}")

import torch
import torch_geometric
print(f"\n✓ PyTorch: {torch.__version__}")
print(f"✓ PyTorch Geometric: {torch_geometric.__version__}")
print(f"✓ CUDA available: {torch.cuda.is_available()}")

Python: 3.10.19 | packaged by conda-forge | (main, Oct 22 2025, 22:29:10) [GCC 14.3.0]
Conda environment: /home/lpatrign/miniconda3/envs/fclga


/home/lpatrign/miniconda3/envs/fclga/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



✓ PyTorch: 2.9.1+cu128
✓ PyTorch Geometric: 2.7.0
✓ CUDA available: True


## Step 2: Preprocessing Pipeline

**Estimated time:** 2-8 hours

In [2]:
# Step 2.1: Generate geometries
!abaqus cae nogui=../src/preprocessing/linear/fclga_generate_geometry.py

Abaqus License Manager checked out the following license:
"cae" from Flexnet server abaqus-research.cc.ic.ac.uk
<56 out of 57 licenses remain available>.


In [59]:
# Step 2.2: Run simulations (Static/Implicit)
!cd .. && python -m src.preprocessing.linear.fclga_run_simulations

ELASTIC CASE: RUNNING ABAQUS SIMULATIONS
Found 500 geometry files to simulate
Input directory: /home/lpatrign/HybridAttentionGNN/data/raw/linear/geometry
Output directory: /home/lpatrign/HybridAttentionGNN/data/raw/linear/simulations

Running 500 simulations with 4 parallel workers

--------------------------------------------------------------------------------
Starting Abaqus job: Plate_0
Starting Abaqus job: Plate_1
Starting Abaqus job: Plate_10
Starting Abaqus job: Plate_100
Analysis initiated from SIMULIA established products
Abaqus JOB Plate_0
Abaqus 2024.HF3
Analysis initiated from SIMULIA established products
Abaqus JOB Plate_100
Abaqus 2024.HF3
Analysis initiated from SIMULIA established products
Abaqus JOB Plate_1
Abaqus 2024.HF3
Analysis initiated from SIMULIA established products
Abaqus JOB Plate_10
Abaqus 2024.HF3
Abaqus License Manager checked out the following licenses:
Abaqus/Standard checked out 5 tokens from Flexnet server abaqus-research.cc.ic.ac.uk.
<533 out of 836 

In [60]:
# Step 2.3: Extract strains (ELEMENT_NODAL position)
!abaqus cae nogui=../src/preprocessing/linear/fclga_extract_results.py

Abaqus License Manager checked out the following license:
"cae" from Flexnet server abaqus-research.cc.ic.ac.uk
<56 out of 57 licenses remain available>.


In [61]:
# Step 2.4: Extract features
!cd .. && python -m src.preprocessing.linear.fclga_extract_features

ELASTIC CASE: EXTRACTING FEATURES FROM .INP FILES
Found 500 .inp files
--------------------------------------------------------------------------------
Processed 1/500: Plate_0.inp (1230 nodes)
Processed 50/500: Plate_49.inp (1210 nodes)
Processed 100/500: Plate_99.inp (1189 nodes)
Processed 150/500: Plate_149.inp (1180 nodes)
Processed 200/500: Plate_199.inp (1243 nodes)
Processed 250/500: Plate_249.inp (1220 nodes)
Processed 300/500: Plate_299.inp (1180 nodes)
Processed 350/500: Plate_349.inp (1225 nodes)
Processed 400/500: Plate_399.inp (1243 nodes)
Processed 450/500: Plate_449.inp (1235 nodes)
Processed 500/500: Plate_499.inp (1207 nodes)
--------------------------------------------------------------------------------
Maximum node count: 1244
Padding to: 1245 (max_node_count + 1)
✓ Node data saved to: /home/lpatrign/HybridAttentionGNN/data/processed/linear/node_gnn_data.pt
✓ Triangulation data saved to: /home/lpatrign/HybridAttentionGNN/data/processed/linear/triangulation_data.pkl


In [62]:
# Step 2.5: Build dataset
!cd .. && python -m src.preprocessing.linear.fclga_build_dataset

ELASTIC CASE: BUILDING STRAIN DATASET
Reading strain files from: /home/lpatrign/HybridAttentionGNN/data/interim/linear/strains
Reading node data from: /home/lpatrign/HybridAttentionGNN/data/processed/linear
Output will be saved to: /home/lpatrign/HybridAttentionGNN/data/processed/linear
Loaded 500 samples from node GNN data
Traceback (most recent call last):
  File "/home/lpatrign/miniconda3/envs/fclga/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/home/lpatrign/miniconda3/envs/fclga/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/home/lpatrign/HybridAttentionGNN/src/preprocessing/linear/fclga_build_dataset.py", line 135, in <module>
    strains_tensor = create_strains_tensor(STRAINS_DIR, node_gnn_data_path)
  File "/home/lpatrign/HybridAttentionGNN/src/preprocessing/linear/fclga_build_dataset.py", line 55, in create_strains_tensor
    raise FileNotFoundError(f"No E11_*.txt files fo

In [63]:
# Step 2.6: Prepare training data
!cd .. && python -m src.preprocessing.linear.fclga_prepare_training_data

ELASTIC CASE: FCLGA Training Data Preparation
Loading preprocessed data...
✓ Loaded 500 graph samples
✓ Loaded 500 strain samples
✓ Loaded geometry data: torch.Size([500, 6])

Processing 500 samples...
Graph is already bidirectional.
  Processing sample 1/500...

First sample statistics:
  Geometry: L=137.45, W=195.07, radius=17.32, displacement=1.0000
  Hole edge nodes: 3.0
  Fixed boundary nodes: 50
  Displaced boundary nodes: 35
  Distance tolerance: 0.8660
  Processing sample 100/500...
  Processing sample 200/500...
  Processing sample 300/500...
  Processing sample 400/500...
  Processing sample 500/500...
✓ Processed all 500 samples

✓ Saved processed data to: /home/lpatrign/HybridAttentionGNN/data/processed/linear/datasets/processed_data.pt

Data Summary
Number of samples: 500
Node features: [x, y, is_hole_edge, is_fixed, is_displaced, displacement_amount]
Node feature shape: torch.Size([1245, 6])
Edge index shape: torch.Size([2, 7084])
Edge attribute shape: torch.Size([7084, 3

## Step 3: Training

With patience=100 to reduce overfitting.

**Estimated time:** 2-10 hours

In [64]:
# Train with hyperparameter optimization
!cd .. && python -m src.training.fclga_train_model \
    --material_type linear \
    --optimize \
    --optuna_trials 50 \
    --epochs 100 \
    --final_epochs 600 

OPTUNA HYPERPARAMETER OPTIMIZATION
Material type: linear
Trials: 50
Epochs per trial: 100
[I 2026-01-08 23:00:20,220] A new study created in memory with name: no-name-b3ec24ab-f3ff-4675-a28f-67c75192baa5

Trial 0: layers=4, hidden=64, lr=1.94e-03
Training:   0%|                                     | 0/100 [00:00<?, ?Epochs/s]
Trial 0 failed with error: Shape mismatch: labels shape torch.Size([4984, 1]) and pred shape torch.Size([4980, 1]) must match
[I 2026-01-08 23:00:20,627] Trial 0 finished with value: inf and parameters: {'num_layers': 4, 'attention_freq': 2, 'batch_size': 4, 'hidden_dim': 64, 'dropout_rate': 0.29733016935699386, 'opt': 'rmsprop', 'opt_decay_step': 46, 'opt_decay_rate': 0.8358975086605895, 'weight_decay': 1.3587805645905419e-06, 'lr': 0.0019447335114736424}. Best is trial 0 with value: inf.

Trial 1: layers=5, hidden=48, lr=5.08e-04
Training:   0%|                                     | 0/100 [00:00<?, ?Epochs/s]
Trial 1 failed with error: Shape mismatch: labels sha

## Step 4: Testing

Using `--training_run` to specify results folder.

**Estimated time:** ~5 minutes

In [65]:
# Test - update timestamp with your training run
!cd .. && python -m src.evaluation.fclga_test \
    --training_run results/linear/training_linear_YYYYMMDD_HHMMSS \
    --material_type linear

AUTO-LOADING FROM TRAINING RUN
Training run directory: results/linear/training_linear_YYYYMMDD_HHMMSS


✗ Error loading from training run: Training run directory not found: results/linear/training_linear_YYYYMMDD_HHMMSS

Please ensure:
  1. The training run directory exists
  2. It contains hyperparameter_optimization_results/best_hyperparameters.json
  3. It contains best_models/*.pt files
Traceback (most recent call last):
  File "/home/lpatrign/miniconda3/envs/fclga/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/home/lpatrign/miniconda3/envs/fclga/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/home/lpatrign/HybridAttentionGNN/src/evaluation/fclga_test.py", line 432, in <module>
    main()
  File "/home/lpatrign/HybridAttentionGNN/src/evaluation/fclga_test.py", line 321, in main
    args = parse_arguments()
  File "/home/lpatrign/HybridAttentionGNN/src/evaluation/fclga_test.py", 

## Step 5: View Results

In [66]:
# Display test result PDF
from IPython.display import IFrame, display
from pathlib import Path

# Update with your training run
training_run = "../results/linear/training_linear_YYYYMMDD_HHMMSS"
pdf_path = Path(training_run) / "test_sample_0_results.pdf"

if pdf_path.exists():
    display(IFrame(str(pdf_path), width=900, height=600))
    print(f"✓ Displaying: {pdf_path}")
else:
    print(f"⚠️ PDF not found: {pdf_path}")
    print("Run Step 4 first to generate results.")
    print(f"Expected location: {pdf_path.resolve()}")

⚠️ PDF not found: ../results/linear/training_linear_YYYYMMDD_HHMMSS/test_sample_0_results.pdf
Run Step 4 first to generate results.
Expected location: /home/lpatrign/HybridAttentionGNN/results/linear/training_linear_YYYYMMDD_HHMMSS/test_sample_0_results.pdf
